# 07 - Result aggregation and best-trial selection

This notebook confirms how a finished study is summarised: best-trial selection
(`study.best_trial`), and the decoding of an indexed-categorical best parameter
performed by the orchestrator's `_decode_phase1_best`
(`pipelines.tuning_pipeline.pipeline`). It exercises the real decoding logic on
a study driven by a cheap synthetic objective.

We confirm that `best_trial` selects the minimum-loss completed trial and that
the raw `features__idx` parameter is decoded back into the channel list.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../..").resolve()))

import numpy             as np
import matplotlib.pyplot as plt
import optuna

import torch

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)

optuna.logging.set_verbosity(optuna.logging.WARNING)

plt.rcParams.update({
    "figure.figsize" : (7.0, 4.2),
    "figure.dpi"     : 110,
    "font.size"      : 10,
    "axes.grid"      : True,
    "grid.alpha"     : 0.3,
})


## A study that samples a real arch space

We sample the `features` indexed-categorical along with a float, scoring trials
so that one specific feature choice is optimal. This produces a best trial
whose raw parameters contain `features__idx`.

In [ ]:
from pipelines.tuning_pipeline.tuners import ParamSampler
from configuration.models_config    import UNetConfig

sampler    = ParamSampler()
arch_space = {'features': UNetConfig.tunable_arch_params()['features']}
feat_choices = arch_space['features']['choices']
BEST_IDX   = 2

def objective(trial):
    sampled = sampler.sample(trial, arch_space)
    extra   = trial.suggest_float('encoder_lr', 1e-5, 1e-2, log=True)
    idx     = feat_choices.index(sampled['features'])
    loss    = (idx - BEST_IDX) ** 2 + (np.log10(extra) - np.log10(1e-3)) ** 2
    return float(loss)

study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=80)

print('best trial number:', study.best_trial.number)
print('best raw params  :', study.best_trial.params)

## Best-trial selection over the value history

Plotting all completed trial values with the best one highlighted confirms
`best_trial` corresponds to the global minimum of the recorded values.

In [ ]:
values  = np.array([t.value for t in study.trials])
best_no = study.best_trial.number

fig, ax = plt.subplots()
ax.scatter(range(len(values)), values, s=20, color='steelblue', label='trial value')
ax.scatter([best_no], [study.best_value], s=160, marker='*', color='red', zorder=5, label='best_trial')
ax.set_xlabel('trial')
ax.set_ylabel('loss')
ax.set_title('Recorded trial values and selected best')
ax.legend()
fig.tight_layout()
plt.show()

print('best value equals min of values:', np.isclose(study.best_value, values.min()))

## Decoding the best parameters

We replicate the orchestrator's `_decode_phase1_best` logic against the model's
lr/arch spaces: any `__idx` parameter whose spec is `indexed_categorical` is
mapped back to the concrete choice. Here we use the arch space directly since
`features` lives there.

In [ ]:
def decode_best(best_trial, space):
    raw     = dict(best_trial.params)
    decoded = {}
    for k, v in raw.items():
        if k.endswith('__idx'):
            param_name = k[:-5]
            spec       = space.get(param_name, {})
            if spec.get('type') == 'indexed_categorical':
                decoded[param_name] = spec['choices'][v]
        else:
            decoded[k] = v
    return decoded

decoded = decode_best(study.best_trial, arch_space)
print('decoded best params:', decoded)
print('expected features  :', feat_choices[BEST_IDX])
print('decoded matches target:', decoded['features'] == feat_choices[BEST_IDX])

## Merged Phase-1 + Phase-2 best parameters

The orchestrator stores the final best config as `{**best_p1, **best_p2}`. We
show that merge on small illustrative dicts so the precedence (Phase-2 wins on
key collisions) is visible.

In [ ]:
best_p1 = {'encoder_lr': 1.2e-3, 'dropout': 0.18, 'decoder_lr': 4e-4}
best_p2 = {'features__idx': BEST_IDX, 'activation': 'gelu', 'dropout': 0.25}

merged = {**best_p1, **best_p2}
for k, v in merged.items():
    src = 'P2' if k in best_p2 else 'P1'
    print(f'  [{src}] {k:16s} = {v}')

print()
print('dropout taken from Phase-2 (override):', merged['dropout'] == best_p2['dropout'])

## Expected visual outcome

The value scatter marks the best trial at the lowest recorded loss, and the
min-equality check prints `True`. Decoding the best parameters recovers the
target feature list (`features` at index 2). The merge listing shows Phase-2
values overriding the colliding `dropout` key from Phase-1.